# Environment and full RDN check

Run this on the computer before opening a model notebook. No dataset training or model accuracy evaluation occurs here.

In [ ]:
from __future__ import annotations
import csv
import hashlib
import json
import math
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import timm
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, roc_auc_score)

# Works when Jupyter starts in the package root or its notebooks directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "lab_code/runtime.py").is_file()
             and (p / "data/full_manifest.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire package and start Jupyter inside it.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab_code.runtime import (
    CLASS_NAMES, FrozenClassifier, FeatureDataset, load_full_data,
    environment_info, select_device, seed_everything, precision_context,
    make_loader, layout_dataset, build_training_loaders, cache_features,
    sha256_file, atomic_json, atomic_checkpoint, save_predictions, save_history,
    plot_history, capture_rng, restore_rng, config_identity,
)
print("Package:", ROOT)


In [ ]:
device = select_device(require_cuda=True)
print(json.dumps(environment_info(device), indent=2))
partitions, identity = load_full_data(ROOT)
print("Verified full split:", {k: len(v) for k, v in partitions.items()})
print("Target: first argmax of compact response >= 83")


## Optional VMamba kernel check

After installing the scan extension, set `CHECK_VMAMBA_KERNEL=True`. This checks forward and backward on small synthetic tensors, including BF16, against the reference scan. It never updates model weights.

In [ ]:
CHECK_VMAMBA_KERNEL = False
if CHECK_VMAMBA_KERNEL:
    from lab_code import vmamba_official as official
    if not official.WITH_SELECTIVESCAN_MAMBA:
        raise RuntimeError("Install selective_scan_cuda first; see START_HERE.md")
    # Shapes mirror VMamba's grouped B/C convention, with a short sequence.
    for dtype in (torch.float32, torch.bfloat16):
        torch.manual_seed(42)
        shape = (1, 8, 16)
        values = [torch.randn(shape, device=device, dtype=dtype) * 0.1,
                  torch.randn(shape, device=device, dtype=dtype) * 0.1,
                  -torch.rand((8, 1), device=device, dtype=torch.float32),
                  torch.randn((1, 4, 1, 16), device=device, dtype=dtype) * 0.1,
                  torch.randn((1, 4, 1, 16), device=device, dtype=dtype) * 0.1,
                  torch.randn(8, device=device, dtype=torch.float32) * 0.1,
                  torch.randn(8, device=device, dtype=torch.float32) * 0.1]
        reference_values = [x.detach().clone().requires_grad_() for x in values]
        kernel_values = [x.detach().clone().requires_grad_() for x in values]
        reference = official.selective_scan_fn(*reference_values, delta_softplus=True,
                                               oflex=True, backend="torch")
        actual = official.selective_scan_fn(*kernel_values, delta_softplus=True,
                                            oflex=True, backend="mamba")
        tolerance = 0.02 if dtype == torch.bfloat16 else 1e-4
        torch.testing.assert_close(actual.float(), reference.float(), rtol=tolerance, atol=tolerance)
        reference.float().square().mean().backward()
        actual.float().square().mean().backward()
        for ref, out in zip(reference_values, kernel_values):
            assert out.grad is not None and torch.isfinite(out.grad).all()
            torch.testing.assert_close(out.grad.float(), ref.grad.float(),
                                       rtol=max(tolerance, 0.02), atol=max(tolerance, 1e-4))
        print("VMamba scan forward/backward passed:", dtype)
else:
    print("Optional VMamba kernel check skipped.")
